# Task 4 EDA — Game Title Images

This notebook describes the supplied training images and checks image integrity without using test labels or fitting a model.

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True
ROOT = Path.cwd()
DATA = ROOT / 'data'
ASSETS = ROOT / 'assets' / 'eda'
ASSETS.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'font.size': 10})
train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
train['class'] = train['class'].str.strip()
classes = ['7-11k', 'fiveN', 'freefried', 'pube', 'roblock']
print('train:', train.shape, 'test:', test.shape)

In [ ]:
counts = train['class'].value_counts().reindex(classes)
counts.rename_axis('class').reset_index(name='count').assign(share=lambda x: x['count'] / len(train)).to_csv(ASSETS / 'class-distribution.csv', index=False)
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(counts.index, counts.values, color=['#4C78A8', '#F58518', '#54A24B', '#E45756', '#B279A2'])
ax.set(title='Task 4 training class distribution', xlabel='Game title', ylabel='Training images')
ax.bar_label(bars, fmt='%d')
fig.tight_layout(); fig.savefig(ASSETS / 'class-distribution.png', bbox_inches='tight'); plt.show()

In [ ]:
def inspect_images(frame, folder, limit=None):
    rows, hashes, failures = [], [], []
    ids = frame['id'].tolist()[:limit] if limit else frame['id'].tolist()
    for image_id in ids:
        path = folder / f'{image_id}.jpg'
        try:
            raw = path.read_bytes()
            with Image.open(path) as im:
                im.verify()
            with Image.open(path) as im:
                rows.append({'id': image_id, 'width': im.width, 'height': im.height, 'mode': im.mode, 'bytes': len(raw)})
            hashes.append(hashlib.md5(raw).hexdigest())
        except Exception as exc:
            failures.append({'id': image_id, 'error': str(exc)})
    return pd.DataFrame(rows), hashes, pd.DataFrame(failures)

# A deterministic 5,000-image integrity sample keeps the notebook practical on Windows.
train_info, train_hashes, failures = inspect_images(train, DATA / 'train', limit=5000)
test_info, _, test_failures = inspect_images(test, DATA / 'test', limit=5000)
integrity = pd.DataFrame([
    {'split': 'train sample', 'rows': len(train), 'files_checked': len(train_info), 'failures': len(failures), 'widths': sorted(train_info.width.unique().tolist()), 'heights': sorted(train_info.height.unique().tolist())},
    {'split': 'test sample', 'rows': len(test), 'files_checked': len(test_info), 'failures': len(test_failures), 'widths': sorted(test_info.width.unique().tolist()), 'heights': sorted(test_info.height.unique().tolist())},
])
integrity.to_csv(ASSETS / 'image-integrity.csv', index=False)
duplicate_count = len(train_hashes) - len(set(train_hashes))
pd.DataFrame([{'split': 'train sample', 'images': len(train_hashes), 'unique_md5': len(set(train_hashes)), 'exact_duplicate_images': duplicate_count}]).to_csv(ASSETS / 'duplicate-summary.csv', index=False)
print(integrity.to_string(index=False)); print('exact duplicate train images:', duplicate_count)

In [ ]:
fig, axes = plt.subplots(len(classes), 4, figsize=(10, 12))
for row, game in enumerate(classes):
    sample = train.loc[train['class'] == game].head(4)
    for col, (_, rec) in enumerate(sample.iterrows()):
        with Image.open(DATA / 'train' / f"{rec['id']}.jpg") as im:
            axes[row, col].imshow(im)
        axes[row, col].axis('off')
        if col == 0: axes[row, col].set_ylabel(game, rotation=90, size=11)
fig.suptitle('Representative training screenshots by game', y=0.995)
fig.tight_layout(); fig.savefig(ASSETS / 'representative-images.png', bbox_inches='tight'); plt.show()

In [ ]:
rng = np.random.default_rng(42)
records = []
for game in classes:
    ids = train.loc[train['class'] == game, 'id'].to_numpy()
    for image_id in rng.choice(ids, size=min(800, len(ids)), replace=False):
        with Image.open(DATA / 'train' / f'{image_id}.jpg') as im:
            arr = np.asarray(im.convert('RGB'), dtype=np.float32) / 255
        records.append({'class': game, 'brightness': float(arr.mean()), 'red': float(arr[:, :, 0].mean()), 'green': float(arr[:, :, 1].mean()), 'blue': float(arr[:, :, 2].mean())})
color = pd.DataFrame(records)
color.groupby('class')[['brightness', 'red', 'green', 'blue']].agg(['mean', 'std']).to_csv(ASSETS / 'color-summary.csv')
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].boxplot([color.loc[color['class'] == game, 'brightness'] for game in classes], labels=classes, showfliers=False)
axes[0].set(title='Brightness by game', ylabel='Mean normalized pixel value')
means = color.groupby('class')[['red', 'green', 'blue']].mean().reindex(classes)
means.plot(kind='bar', ax=axes[1], color=['#E45756', '#54A24B', '#4C78A8'])
axes[1].set(title='Mean RGB values by game', xlabel='Game title', ylabel='Mean normalized channel value'); axes[1].legend(title='Channel')
fig.tight_layout(); fig.savefig(ASSETS / 'color-distributions.png', bbox_inches='tight'); plt.show()